# PART 3 — 画像はどう分類する？
## 実習3：Dry Beanで7種類の豆を分類しよう

**Image → Feature Extraction → Multiclass Classification**

この実習で使うのは **Hand-crafted Features（人が設計した特徴量）** です。

Google Colabで開き、上から順にセルを実行してください。CPUで実行できます。データ取得時にはインターネット接続が必要です。

### 画像から分類までの流れ
1. **Image：** 豆を撮影する。
2. **Segmentation：** 画像から豆の領域を取り出す。
3. **Feature Extraction：** 豆の面積・周囲長・形などを数値にする。
4. **Multiclass Classification：** 数値を入力し、7種類のどれかを予測する。

**今回の公開データには抽出済みの特徴量が入っています。元画像の読み込みや画像からの領域抽出は、このノートブックでは行いません。** 抽出済みデータを使って、特徴量の意味と分類の流れを学びます。

Hand-craftedとは、人が特徴量の定義・計算方法を設計することです。値はプログラムで自動計算できます。分類の規則は機械学習で学びます。

## 準備
UCI Machine Learning Repository のデータを直接読み込むために、`ucimlrepo` を用意します。


In [ ]:
%pip -q install ucimlrepo


## 1 ライブラリを読み込む
ペンギン・乳房腫瘍の実習と同じ決定木を使います。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


## 2 データを読み込む
UCI Machine Learning Repository から **Dry Bean Dataset (ID=602)** を直接読み込みます。
1行が豆1粒、`Class` 列が豆の種類です。取得時にはインターネット接続が必要です。


In [ ]:
# UCI Machine Learning Repository から Dry Bean Dataset を読み込む
dry_bean = fetch_ucirepo(id=602)

# 16個の特徴量
X = dry_bean.data.features.copy()

# 豆の種類（7クラス）
y = dry_bean.data.targets.copy()

# 特徴量とクラスを1つのDataFrameにまとめる
df = pd.concat([X, y], axis=1)

print("行数と列数:", df.shape)
display(df.head())


## 3 Hand-crafted Featuresを確認する

| 特徴量 | 意味 |
|---|---|
| Area | 豆の領域の面積（画素数） |
| Perimeter | 豆の輪郭の長さ |
| MajorAxisLength | 長軸の長さ |
| MinorAxisLength | 短軸の長さ |
| AspectRatio | 長軸と短軸の比。大きいほど細長い |
| Roundness | 丸さの指標：4π × 面積 ÷ 周囲長² |

全部で16個の特徴量があり、色の値や画像そのものは入力に含めません。

次のセルでは、すでに与えられた面積と周囲長から丸さを計算します。画像処理そのものではなく、特徴量の計算式を確かめる練習です。


In [ ]:
roundness_check = 4 * np.pi * df["Area"] / df["Perimeter"] ** 2

display(pd.DataFrame({
    "データの丸さ": df["Roundness"],
    "式から計算した丸さ": roundness_check
}).head())

## 4 欠損値・重複を確認する
同じ行が学習用とテスト用の両方に入ることを避けるため、完全に同じ行は分割前に1行にまとめます。欠損行も確認して除きます。行番号は元データのものを残します。

In [ ]:
print("欠損値の数:", df.isnull().sum().sum())
print("重複行の数:", df.duplicated().sum())
df = df.dropna().drop_duplicates()
print("処理後の行数:", len(df))
print(df["Class"].value_counts())


In [ ]:
print(df.columns.tolist())

## 5 特徴量を図で見る
面積と細長さの2つだけで、どの種類が分かれやすいでしょうか。重なる種類も探してください。図は2特徴、分類モデルは16特徴を使います。図中の英語表記は文字化けを避けるためです。

In [ ]:
plt.figure(figsize=(9, 6))
sns.scatterplot(data=df, x="Area", y="AspectRatio", hue="Class",
                s=12, alpha=0.35)
plt.xlabel("Area (pixels)")
plt.ylabel("Aspect ratio")
plt.title("Dry Bean: area and shape")
plt.tight_layout()
plt.show()


## 6 入力 X と正解 y を分ける
Class以外の16列をXに、種類の名前をyに入れます。文字列のクラス名をそのまま使えます。

In [ ]:
X = df.drop(columns=["Class"])
y = df["Class"]
print("特徴量の数:", X.shape[1])
print("クラスの数:", y.nunique())


## 7 学習用とテスト用に分割する
約80%で学習し、約20%で評価します。stratify=yで各種類の比率をできるだけ保ちます。

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("学習用:", len(X_train))
print("テスト用:", len(X_test))


## 8 決定木で学習し、予測する
7種類を分類するため、今回は木の深さの上限をあらかじめ5に設定します。最適な値という意味ではありません。テスト結果を見ながら設定を選ぶことは、この実習では行いません。決定木では標準化の処理は省略できます。

In [ ]:
model = DecisionTreeClassifier(max_depth=5, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("正答率:", round(accuracy_score(y_test, y_pred) * 100, 1), "%")


## 9 混同行列で間違い方を見る
行が実際の種類、列が予測した種類です。対角線上は正解、対角線以外は誤分類です。ペンギンの3クラス、乳房腫瘍の2クラスから、今回は7クラスへ増えました。

In [ ]:
classes = model.classes_
cm = confusion_matrix(y_test, y_pred, labels=classes)
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=classes, yticklabels=classes)
plt.xlabel("Predicted class")
plt.ylabel("Actual class")
plt.title("Confusion matrix")
plt.tight_layout()
plt.show()


## 10 種類ごとの成績を見る

- **precision（適合率）：** その種類と予測した中で、正解だった割合。
- **recall（再現率）：** 実際にその種類だった中で、正しく見つけた割合。
- **f1-score：** 適合率と再現率の調和平均。
- **support：** テストデータ内の、その種類の件数。

7クラスそれぞれを注目するクラスとして評価します。macro avgは7クラスを同じ重みで平均し、weighted avgは件数で重みを付けて平均します。値は0〜1で表示され、0.90は90%です。

In [ ]:
print(classification_report(y_test, y_pred, digits=3, zero_division=0))


## 11 未学習の1粒を予測する
テスト用データから1粒だけ取り出します。これは新しく撮影した豆ではなく、学習に使わなかったデータです。
`iloc[[0]]`は先頭の1行を表の形で取り出し、`iloc[0]`は先頭の値を取り出します。

In [ ]:
one_bean = X_test.iloc[[0]]
prediction = model.predict(one_bean)
print("予測:", prediction[0])
print("実際:", y_test.iloc[0])


## 確認問題
1. 今回、モデルに入力したものは画像ですか。それとも数値ですか。
2. Hand-crafted Featuresの「人が設計するもの」は何ですか。
3. 面積が同じでも、細長い豆と丸い豆を区別するにはどんな特徴が役立ちそうですか。
4. 混同行列から、間違えやすい種類の組合せを1つ挙げてください。
5. 再現率が最も低い種類は何ですか。
6. 新しく撮影した豆を分類するには、分類モデルに渡す前に何が必要ですか。

### 提出するもの
正答率、混同行列、上の6問への回答。

### まとめ
画像を数値特徴量で表すことで、表データで使った分類器を画像由来の問題にも使えます。新しい画像にも、学習時と同じ特徴の定義・計算方法・撮影条件や尺度への配慮が必要です。

深層学習では、画像から分類に役立つ特徴表現も学習します。今回の特徴量は人が定義した形状の数値であり、分類器が学ぶのはその数値と種類の関係です。

## 出典
- Dry Bean [Dataset] (2020), UCI Machine Learning Repository, https://doi.org/10.24432/C50S4B （CC BY 4.0）
- https://archive.ics.uci.edu/dataset/602/dry+bean
- Köklü, M. & Özkan, İ. A. (2020), Multiclass classification of dry beans using computer vision and machine learning techniques.
- [scikit-learn：決定木](https://scikit-learn.org/stable/modules/tree.html)
- [scikit-learn：classification_report](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html)

作成時の確認環境：scikit-learn 1.8.0。実行環境やデータの版が異なると結果が変わる場合があります。